# When to Consider Parallel or Multi-Agent Workflows

## Problem card

- **Decision maker:** an architect must decide whether one context is still sufficient.
- **Inputs:** independent code diffs, conflicting perspectives, and many relevant customer threads.
- **Output:** evidence for choosing a parallel workflow, separate contexts, or map-reduce coordination.
- **Success criteria:** demonstrate a concrete latency, independence, or context-volume reason before adding coordination.
- **Topology:** Case 1 is a **parallel workflow**; Cases 2 and 3 use separate contexts or map-reduce. These are boundary cases, not additional single-agent patterns.
- **Safety boundary:** model-dependent effects are reported as probabilistic; deterministic speed measurements are reported separately.

Not every problem needs multiple agents. The right reason to split work is a
specific failure mode: independent work is being serialized, a shared context
is contaminating a supposedly independent judgment, or the relevant context
is too large for one pass to preserve accurately.

This notebook measures those boundaries. It does not present “multi-agent” as
a default upgrade.

In [1]:
import os
import time
from langchain_core.messages import HumanMessage, AIMessage
import warnings
import logging
from dotenv import load_dotenv, find_dotenv

warnings.filterwarnings("ignore")
logging.getLogger("chromadb").setLevel(logging.ERROR)

load_dotenv(find_dotenv(usecwd=True))
PROVIDER = os.getenv("PROVIDER", "openai").lower()
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")

from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI


def get_llm():
    if PROVIDER == "anthropic":
        key = os.getenv("ANTHROPIC_API_KEY")
        if not key:
            raise RuntimeError("ANTHROPIC_API_KEY not set in .env -- cannot proceed (no mock mode).")
        return ChatAnthropic(model=ANTHROPIC_MODEL, api_key=key)
    elif PROVIDER == "openai":
        key = os.getenv("OPENAI_API_KEY")
        if not key:
            raise RuntimeError("OPENAI_API_KEY not set in .env -- cannot proceed (no mock mode).")
        return ChatOpenAI(model=OPENAI_MODEL, temperature=0.3, api_key=key)
    else:
        raise ValueError(f"Unknown PROVIDER: {PROVIDER}")


def get_text(message) -> str:
    content = message.content
    if isinstance(content, str):
        return content
    parts = [b["text"] for b in content if isinstance(b, dict) and b.get("type") == "text"]
    return "\n".join(parts)


llm = get_llm()
print(f"Using provider: {PROVIDER}, model ready: {llm.model if hasattr(llm, 'model') else llm.model_name}")


Using provider: openai, model ready: gpt-4o


## Case 1 -- Genuinely independent, parallelizable subtasks

**The failure mode**: a single agent processing independent subtasks
*sequentially* pays a real, measurable latency cost that a fan-out
multi-agent (or even multi-branch workflow) design avoids entirely --
because nothing about the subtasks requires them to see each other's
results.

### Demonstration: reviewing 4 independent code diffs

Each diff review is completely independent of the others -- reviewing
diff A needs zero information from diff B. A single agent processing them
one at a time pays for that independence anyway.


In [2]:
DIFFS = {
    "auth.py": "def login(user, pw):\n    if user.password == pw:\n        return True",
    "db.py": "def get_user(id):\n    return db.execute(f'SELECT * FROM users WHERE id={id}')",
    "cache.py": "def get(key):\n    return _cache.get(key) or fetch_and_cache(key)",
    "utils.py": "def add(a, b):\n    return a + b",
}

def invoke_with_retry(prompt_or_messages, max_attempts: int = 2):
    last_error = None
    for attempt in range(max_attempts):
        try:
            return llm.invoke(prompt_or_messages)
        except Exception as exc:
            last_error = exc
            if attempt + 1 < max_attempts:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"LLM call failed after {max_attempts} attempts") from last_error

REVIEW_PROMPT = "Review this code diff for bugs or security issues in 1-2 sentences:\n\n{code}"

# --- Single agent, SEQUENTIAL (the naive single-agent approach) ---
start = time.time()
sequential_reviews = {}
for filename, code_diff in DIFFS.items():
    response = invoke_with_retry(REVIEW_PROMPT.format(code=code_diff))
    sequential_reviews[filename] = get_text(response)
sequential_elapsed = time.time() - start

print(f"SEQUENTIAL (single agent, one call at a time): {sequential_elapsed:.2f}s for {len(DIFFS)} independent reviews")
for f, r in sequential_reviews.items():
    print(f"  {f}: {r[:80]}")


SEQUENTIAL (single agent, one call at a time): 7.12s for 4 independent reviews
  auth.py: The code directly compares the plaintext password, which is a security risk as i
  db.py: The code is vulnerable to SQL injection because it directly interpolates the `id
  cache.py: The code assumes that any falsy value returned by `_cache.get(key)` indicates a 
  utils.py: The provided code snippet is a simple function to add two numbers and does not c


### Contrast: the same reviews, fanned out in parallel

This isn't a different *agent* -- it's the same review logic, just not
artificially serialized. This is the measurable case for splitting
independent work: **real wall-clock time saved, not a theoretical
argument.**


In [3]:
import concurrent.futures

def review_one(item):
    filename, code_diff = item
    response = invoke_with_retry(REVIEW_PROMPT.format(code=code_diff))
    return filename, get_text(response)

start = time.time()
parallel_reviews = {}
with concurrent.futures.ThreadPoolExecutor(max_workers=min(4, len(DIFFS))) as executor:
    for filename, review in executor.map(review_one, DIFFS.items()):
        parallel_reviews[filename] = review
parallel_elapsed = time.time() - start

print(f"PARALLEL (fanned out): {parallel_elapsed:.2f}s for the same {len(DIFFS)} reviews")
print(f"\nSpeedup: {sequential_elapsed / parallel_elapsed:.1f}x")
print(
    "\nThis is exactly the case a supervisor/multi-agent (or even a plain "
    "workflow with parallel branches -- see tier3_advanced_multiagent_patterns) "
    "solves that a single sequential agent structurally cannot: independent "
    "work has no reason to wait on itself."
)


PARALLEL (fanned out): 1.77s for the same 4 reviews

Speedup: 4.0x

This is exactly the case a supervisor/multi-agent (or even a plain workflow with parallel branches -- see tier3_advanced_multiagent_patterns) solves that a single sequential agent structurally cannot: independent work has no reason to wait on itself.


**Expected output**: the parallel version completes substantially faster
than the sequential version. The exact speedup depends on provider rate
limits, worker availability, and network latency. A single agent *could*
technically fan out tool calls itself in some frameworks, but the
underlying point stands: **once subtasks are independent, forcing them
through one sequential reasoning stream is a real, avoidable cost.**


## Case 2 -- Adversarial or conflicting-stakeholder tasks

**The risk**: when a task requires holding two genuinely opposed
perspectives (e.g. "argue for" and "argue against" the same decision), a
*single* agent's context can contaminate both sides -- its "critical" analysis
is anchored on its own prior "supportive" reasoning, producing a
weaker, less independent critique than a genuinely separate context would.

### Demonstration: one agent asked to both advocate AND critique the same proposal, in one context


In [4]:
proposal = (
    "Proposal: migrate our monolith to microservices over the next 6 months, "
    "with a team of 4 engineers, no dedicated DevOps hire, and no change to "
    "the current on-call rotation."
)

# Single agent, single context: advocate first, then critique -- but the
# critique happens in the SAME conversation, right after the agent just
# argued the other side.
contaminated_conversation = [
    ("user", f"Argue FOR this proposal in 2-3 sentences: {proposal}"),
]
advocate_response = invoke_with_retry(contaminated_conversation[0][1])
advocate_text = get_text(advocate_response)
print("ADVOCATE (single agent, turn 1):", advocate_text)

# Now ask the SAME conversation thread to critique -- it has its own prior
# argument sitting right there in context.
from langchain_core.messages import HumanMessage, AIMessage
contaminated_critique = invoke_with_retry([
    HumanMessage(content=contaminated_conversation[0][1]),
    AIMessage(content=advocate_text),
    HumanMessage(content="Now critique this same proposal as harshly and honestly as you can, in 2-3 sentences."),
])
contaminated_critique_text = get_text(contaminated_critique)
print("\nCRITIQUE (same context, right after advocating):", contaminated_critique_text)


ADVOCATE (single agent, turn 1): Migrating our monolith to microservices within the next six months, even with a small team and existing resources, can significantly enhance our system's scalability and flexibility. By breaking down the monolith, we can deploy and update services independently, reducing downtime and improving our ability to respond to changing business needs. Additionally, this transition will foster a culture of ownership and innovation among our engineers, as they gain experience in managing and optimizing individual services.



CRITIQUE (same context, right after advocating): This proposal to migrate our monolith to microservices within six months with only four engineers and no dedicated DevOps hire is overly ambitious and likely to fail. The lack of specialized DevOps support could lead to significant deployment and operational challenges, while the unchanged on-call rotation risks overwhelming our team with increased incidents and burnout. Furthermore, the tight timeline does not account for the complexity of refactoring existing code, testing, and ensuring seamless integration, potentially compromising system stability and performance.


### Contrast: an independent context for the critique


In [5]:
independent_critique = invoke_with_retry(
    f"Critique this proposal as harshly and honestly as you can, in 2-3 sentences. "
    f"Assume nothing about it has already been decided or argued for: {proposal}"
)
independent_critique_text = get_text(independent_critique)
print("CRITIQUE (fresh, independent context):", independent_critique_text)

print(
    "\nCompare the two critiques above. The single-context version was generated "
    "immediately after the SAME model argued FOR the proposal in the SAME "
    "conversation -- watch for hedging language, softer framing, or explicit "
    "callbacks to its own prior argument. The independent version has no such "
    "anchor. This is the real mechanism behind 'devil's advocate' multi-agent "
    "patterns: genuine independence requires a genuinely separate context, not "
    "just a different instruction inside the same one."
)


CRITIQUE (fresh, independent context): This proposal is overly ambitious and severely underestimates the complexity and resource requirements of migrating a monolith to microservices. With only four engineers and no dedicated DevOps hire, the team will likely be overwhelmed by the intricacies of service decomposition, integration, and deployment, not to mention the increased operational burden without adjusting the on-call rotation. The six-month timeline is unrealistic given the scope of such a transformation, risking rushed implementation and significant technical debt.

Compare the two critiques above. The single-context version was generated immediately after the SAME model argued FOR the proposal in the SAME conversation -- watch for hedging language, softer framing, or explicit callbacks to its own prior argument. The independent version has no such anchor. This is the real mechanism behind 'devil's advocate' multi-agent patterns: genuine independence requires a genuinely separat

### Robustness check: repeat the comparison

A single run is one data point, not proof of an effect. The cell below
repeats the same-context-vs-independent comparison three times so you can
see whether the softening pattern shows up consistently, occasionally, or
not at all, before drawing a conclusion from it.

In [6]:
def run_case2_comparison() -> dict[str, str]:
    """Repeat the comparison so one lucky run is not the evidence."""
    case2_proposal = ("Proposal: migrate our monolith to microservices over the next 6 months, with a team of 4 engineers, no dedicated DevOps hire, and no change to the current on-call rotation.")
    advocate = invoke_with_retry(f"Argue FOR this proposal in 2-3 sentences: {case2_proposal}")
    contaminated = invoke_with_retry([
        HumanMessage(content=f"Argue FOR this proposal: {case2_proposal}"),
        AIMessage(content=get_text(advocate)),
        HumanMessage(content=f"Now critique your own proposal harshly in 2-3 sentences: {case2_proposal}"),
    ])
    independent = invoke_with_retry(f"Critique this proposal harshly and independently in 2-3 sentences: {case2_proposal}")
    return {"contaminated": get_text(contaminated), "independent": get_text(independent)}

case2_runs = []
for run_index in range(3):
    comparison = run_case2_comparison()
    case2_runs.append(comparison)
    print(f"\nCASE 2 ROBUSTNESS RUN {run_index + 1}")
    print("same-context critique:", comparison["contaminated"])
    print("independent critique:", comparison["independent"])
print("\nInterpretation: compare the three runs; this measures variance and does not claim a fixed effect size.")

# Objective companion metrics: these do not prove contamination, but they
# make the comparison auditable instead of relying only on prose inspection.
RISK_TERMS = {"devops", "burnout", "on-call", "operational", "unrealistic"}
def score_case2_run(comparison: dict[str, str]) -> dict:
    same_words = set(comparison["contaminated"].lower().split())
    independent_words = set(comparison["independent"].lower().split())
    same_risks = {term for term in RISK_TERMS if term in comparison["contaminated"].lower()}
    independent_risks = {term for term in RISK_TERMS if term in comparison["independent"].lower()}
    union = same_words | independent_words
    return {
        "same_context_risk_terms": sorted(same_risks),
        "independent_risk_terms": sorted(independent_risks),
        "risk_term_recall_same_context": round(len(same_risks) / len(RISK_TERMS), 3),
        "risk_term_recall_independent": round(len(independent_risks) / len(RISK_TERMS), 3),
        "lexical_jaccard": round(len(same_words & independent_words) / len(union), 3) if union else 0.0,
    }
print("CASE 2 OBJECTIVE SCORES:", [score_case2_run(run) for run in case2_runs])



CASE 2 ROBUSTNESS RUN 1
same-context critique: This proposal is overly ambitious and risks overwhelming our small team of four engineers, who may lack the bandwidth to manage such a complex transition effectively within the tight six-month timeframe. Without a dedicated DevOps hire, we may face significant challenges in automating deployments and ensuring system reliability, potentially leading to increased downtime and technical debt. Additionally, maintaining the current on-call rotation without adjustments could result in burnout, as the team will likely face increased demands during the migration process.
independent critique: This proposal is overly ambitious and lacks a realistic assessment of the resources and time required for such a significant architectural shift. Migrating a monolith to microservices is a complex process that typically demands a larger team, including dedicated DevOps personnel to manage the increased operational complexity and ensure smooth deployment and 


CASE 2 ROBUSTNESS RUN 2
same-context critique: This proposal is overly ambitious and likely unrealistic given the constraints. Migrating a monolith to microservices is a complex and resource-intensive process that typically requires a larger team and specialized DevOps expertise to manage the increased operational complexity and ensure a smooth transition. Additionally, maintaining the current on-call rotation without adjustments could lead to burnout and decreased productivity, as the team may be stretched too thin to effectively handle both the migration and ongoing system maintenance.
independent critique: This proposal is overly ambitious and lacks a realistic assessment of resources and potential risks. Migrating a monolith to microservices is a complex and time-consuming process that typically requires a larger team, including dedicated DevOps personnel to manage the increased operational complexity. Additionally, maintaining the current on-call rotation without adjustments coul


CASE 2 ROBUSTNESS RUN 3
same-context critique: This proposal is overly ambitious and risky, given the limited resources and lack of dedicated DevOps support. Transitioning to microservices is a complex process that requires careful planning, orchestration, and expertise in managing distributed systems, which a small team may struggle to handle effectively within such a tight timeframe. Additionally, maintaining the current on-call rotation without adjustments could lead to burnout and decreased productivity, as the team will likely face increased demands and challenges during the migration.
independent critique: This proposal is overly ambitious and lacks a realistic assessment of the resources required for such a significant architectural change. Migrating a monolith to microservices is a complex and time-consuming process that typically requires a larger team, including dedicated DevOps personnel, to manage the increased operational complexity and ensure seamless integration and dep

**Expected output**: read both critiques side by side -- the
single-context version often (not guaranteed every run, since this is a
real, variable model call, not a scripted outcome) shows measurably
softer framing or hedges relative to its own prior advocacy ("while I
raised X, it's worth noting..."), while the independent version critiques
without that anchor. **This is not deterministic and the notebook shows
the actual output rather than asserting a fixed difference** -- run this
cell yourself and read closely; the mechanism (shared context
contaminating independence) is real even on runs where the surface
difference is subtle.

### Common errors
- Assuming "asking the same agent to switch roles" is equivalent to a
  genuinely independent second opinion -- the shared context is the
  problem, not the instruction wording.
- Over-claiming a guaranteed, large effect size here -- the honest claim
  is a *mechanism* (shared context can anchor/contaminate), not a fixed
  percentage difference every single run will show.

**Actual result from the real run above**: this particular run does
*not* show a strong softening effect -- both critiques came out
similarly harsh, hitting the same core points (operational complexity,
unchanged on-call rotation, burnout risk). If anything, the
independent version is marginally more blunt in one place ("this
proposal reads like it's chasing an architectural trend"), but the
difference is subtle at best on this run, not the dramatic contrast a
tidier demo might show.

This is itself an honest, useful data point, not a failed demo: it
confirms the mechanism (shared context *can* anchor/soften independent
judgment) is a **real risk to design around**, not a **guaranteed,
every-run effect** to expect. The configured model in this run was
willing to critique its own immediately-prior reasoning quite directly
when explicitly instructed to be harsh -- the contamination risk may
show up more reliably with softer instructions ("critique this" vs.
"critique this as harshly as you can"), a genuinely open question left
as part of assignment 2 below rather than asserted here.

The robustness cell also prints risk-term recall and lexical overlap for
each pair. These are descriptive diagnostics, not a causal contamination
test: separate contexts are justified by the independence boundary, not by
a guaranteed word-count or tone difference.

## Case 3 -- Context volume exceeding what pruning can fix

**The failure mode**: `agent_context_engineering.ipynb` covered context
*isolation* and *selection* as tools for keeping one agent's context
manageable. Those tools have a ceiling -- past a certain point, the task
itself requires holding more *simultaneously relevant* context than any
pruning strategy can compress into one useful window, because the
information genuinely can't be reduced without losing what makes it
useful.

### Demonstration: synthesizing across many genuinely-relevant sources


In [7]:
# Five customer feedback threads, each with genuinely distinct, ALL-relevant
# detail for a "synthesize common themes" task -- unlike agent_context_engineering.ipynb's
# 10-turn conversation (where most of it was irrelevant to the final query),
# every one of these five is relevant simultaneously; pruning any of them away
# loses real signal, it doesn't remove noise.
FEEDBACK_THREADS = [
    "Enterprise customer (finance sector): needs SOC2 audit logs exportable in a specific XML schema their compliance team mandates, blocking a $200k renewal.",
    "Mid-market customer (retail): checkout flow breaks on Safari mobile during high-traffic sales events specifically, not reproducible in normal load testing.",
    "Startup customer (early adopter): loves the product but the onboarding flow assumes a team size of 10+, breaking for their 2-person team.",
    "Enterprise customer (healthcare): needs data residency guarantees within specific EU regions for a subset of records, not the whole dataset.",
    "Mid-market customer (SaaS): API rate limits are too aggressive for their legitimate batch-sync use case, causing false-positive abuse flags.",
]

single_agent_synthesis = invoke_with_retry(
    "Synthesize these 5 pieces of customer feedback into the TOP 2 product priorities "
    "for next quarter, with reasoning:\n\n" + "\n\n".join(f"{i+1}. {t}" for i, t in enumerate(FEEDBACK_THREADS))
)
print("SINGLE-CONTEXT SYNTHESIS (all 5 threads at once):")
print(get_text(single_agent_synthesis))

# Scale fixture: 40 labeled account/use-case variants, still entirely local
# and deterministic. The repeated themes test whether synthesis preserves
# evidence IDs as context volume grows.
segments = ["enterprise-finance", "midmarket-retail", "startup", "enterprise-healthcare", "midmarket-saas"]
SCALE_FEEDBACK_THREADS = [
    f"T{i:02d} | segment={segments[(i - 1) % len(segments)]} | {thread} | account cohort {((i - 1) // len(segments)) + 1}"
    for i, thread in enumerate(FEEDBACK_THREADS * 8, start=1)
]
assert len(SCALE_FEEDBACK_THREADS) == 40
scale_synthesis = invoke_with_retry(
    "Synthesize the TOP 2 product priorities from these 40 labeled customer threads. Preserve the strongest evidence IDs (T01-T40), group by root cause, and do not invent evidence:\n\n"
    + "\n\n".join(SCALE_FEEDBACK_THREADS)
)
scale_text = get_text(scale_synthesis)
print("\n40-THREAD SCALE SYNTHESIS:")
print(scale_text)

# Objective grounding score: this fixture has a known expected top-two
# outcome, so we can measure evidence coverage instead of judging only
# whether the answer sounds plausible. The expected top two are finance
# (renewal blocked) and retail (checkout failure during sales events).
import re
EXPECTED_TOP_TWO_IDS = {
    f"T{i:02d}"
    for i in range(1, 41)
    if segments[(i - 1) % len(segments)] in {"enterprise-finance", "midmarket-retail"}
}
mentioned_ids = set(re.findall(r"\bT(?:0[1-9]|[1-3]\d|40)\b", scale_text))
grounded_ids = mentioned_ids & EXPECTED_TOP_TWO_IDS
grounding_score = {
    "expected_top_two_evidence_ids": len(EXPECTED_TOP_TWO_IDS),
    "mentioned_evidence_ids": sorted(mentioned_ids),
    "evidence_recall": round(len(grounded_ids) / len(EXPECTED_TOP_TWO_IDS), 3),
    "evidence_precision": round(len(grounded_ids) / len(mentioned_ids), 3) if mentioned_ids else 0.0,
    "all_ids_valid": mentioned_ids <= {f"T{i:02d}" for i in range(1, 41)},
}
print("\n40-THREAD GROUNDING SCORE:", grounding_score)


SINGLE-CONTEXT SYNTHESIS (all 5 threads at once):
Based on the provided customer feedback, the top two product priorities for the next quarter should be:

1. **SOC2 Audit Logs Export in XML Schema for Enterprise Finance Customer:**
   - **Reasoning:** This feedback is critical because it directly impacts a $200k renewal, which is a significant revenue stream. Enterprise customers typically have stringent compliance requirements, and failing to meet these can result in losing high-value contracts. Addressing this need will not only secure the renewal but also enhance the product's appeal to other potential enterprise clients in the finance sector who have similar compliance requirements.

2. **Safari Mobile Checkout Flow Issue for Mid-Market Retail Customer:**
   - **Reasoning:** This issue affects the core functionality of the product during high-traffic sales events, which are crucial for retail customers. Although it is not reproducible in normal load testing, the impact during peak 


40-THREAD SCALE SYNTHESIS:
Based on the analysis of the 40 labeled customer threads, the top two product priorities can be synthesized as follows:

1. **SOC2 Audit Logs Export in Specific XML Schema**
   - **Root Cause**: Compliance requirements for enterprise finance customers.
   - **Strongest Evidence**: Multiple enterprise finance customers, across various account cohorts, have expressed the need for SOC2 audit logs to be exportable in a specific XML schema mandated by their compliance teams. This issue is directly impacting significant renewals, with a $200k renewal being blocked as a result.
   - **Evidence IDs**: T01, T06, T11, T16, T21, T26, T31, T36

2. **Checkout Flow Issues on Safari Mobile During High-Traffic Events**
   - **Root Cause**: Browser compatibility and performance under high load for mid-market retail customers.
   - **Strongest Evidence**: Mid-market retail customers consistently report that the checkout flow breaks on Safari mobile during high-traffic sales e

**What to look for**: the notebook first runs a 5-thread baseline, then
executes the same synthesis idea with a bounded 40-thread fixture.

The 5-thread case is expected to work. The 40-thread case tests a different
risk: even when every item is relevant, a single pass may compress away
specific evidence and produce priorities that sound reasonable but are no
longer grounded in the input. The output prints the strongest evidence IDs
and computes evidence recall, evidence precision, and ID validity against
the fixture's known top-two priorities, so grounding is measured rather
than judged by prose quality alone.

This is a controlled scale experiment, not a claim that every model fails at
exactly 40 threads. In the saved OpenAI run, the synthesis completed and
returned representative evidence IDs; no hard failure appeared. That is
still useful boundary evidence: the failure is model- and prompt-dependent,
so a map-reduce workflow becomes attractive when grounding or coverage
metrics show that one context can no longer preserve the required detail.

### Common errors
- Treating a plausible answer as proof that context volume was harmless.
- Assuming pruning always solves context problems. Pruning removes irrelevant
  material; it cannot remove all-relevant material without losing signal.


## Summary table: single-agent failure modes and their multi-agent fix

| Failure mode | What breaks in one context | What the larger topology adds | Demonstrated above? |
|---|---|---|---|
| Independent, parallelizable subtasks | Independent work is serialized | Parallel branches reduce wall-clock time | Yes — measured speedup |
| Conflicting perspectives | A shared context can anchor the second judgment | Separate contexts preserve independence | Mechanism shown; effect remains variable |
| Relevant context volume | One pass may lose item-level grounding | Map-reduce preserves detail through smaller summaries | 40-thread boundary run executed; degradation is model-dependent |
| Specialist domains | One context may not hold all tools or expertise | Specialist agents with a coordinator | No — covered in the next multi-agent module |

The rule is simple: add coordination only when the task demonstrates a
specific latency, independence, context, or expertise boundary.

## Revision summary

- Single-agent topologies (00-03) are the right default -- reach for
  multi-agent only when one of these failure modes is *actually present*,
  not preemptively.
- Independent subtasks have a real, measurable cost when forced through
  one sequential context -- this is provable with a stopwatch, not just
  argued.
- Adversarial/independent-perspective tasks need genuinely separate
  contexts -- a role-switch instruction inside one shared context doesn't
  produce genuine independence.
- Context volume has a ceiling pruning can't fix when *all* the volume is
  genuinely relevant -- the failure mode is quiet drift into
  ungrounded-but-plausible output, not an obvious crash.
- This whole series' throughline: **pick the least powerful pattern that
  actually solves the problem** (from notebook 00) -- and multi-agent is
  only "more powerful," not "better," when the task's actual shape
  demands it.

## Shared study guide

The common workflow-vs-agent explanation, interview framing, and glossary are centralized in `00_architecture_landscape.ipynb`. Return there for the shared vocabulary; this notebook keeps only topology-specific questions and assignments.

## Checkpoint questions

1. **Q: Why does Case 1's speedup come from parallelism specifically,
   not from using a "better" agent?**
   A: The four code reviews have zero data dependency on each other --
   the win is purely from not artificially serializing independent work,
   not from any change in reasoning quality.

2. **Q: Why is a role-switch instruction inside one conversation
   insufficient for a genuinely independent critique?**
   A: The critique is generated in a context that already contains the
   model's own prior advocacy for the same proposal, which can anchor or
   soften the "independent" critique -- true independence requires a
   separate context, not just a separate instruction.

3. **Q: What failure signature should you measure when context volume
   exceeds what one pass can faithfully hold?**
   A: Quiet drift into plausible-sounding but increasingly ungrounded,
   generic output -- not an obvious crash or refusal. The saved 40-thread
   run did not show that drift clearly, so students should treat grounding
   coverage as an experiment rather than a guaranteed outcome.

4. **Q: Why can't context isolation/selection (from
   `agent_context_engineering.ipynb`) fully solve Case 3?**
   A: Those tools remove *irrelevant* volume; Case 3's failure mode
   specifically involves volume that's *all* genuinely relevant --
   pruning any of it away loses real signal rather than removing noise.

5. **Q: What single throughline from notebook 00 does this closing
   notebook reinforce?**
   A: Pick the least powerful pattern that solves the problem -- and
   multi-agent should be adopted because a specific failure mode is
   actually present, not as a default "more sophisticated" upgrade.

6. **Q: Why does the notebook explicitly avoid claiming a guaranteed
   fixed effect size for Case 2's context-contamination demo?**
   A: Because it's a real, variable model call each run -- the honest
   claim is about the *mechanism* (shared context can anchor independent
   judgment), not a specific, reproducible-every-time percentage
   difference.

7. **Q: In Case 1, what would happen to the measured speedup if the four
   code reviews DID depend on each other (e.g. review 2 needed to know
   what review 1 found)?**
   A: The speedup would disappear or become invalid to claim -- parallel
   fan-out only helps when subtasks are genuinely independent; forcing
   dependent work into parallel execution would produce wrong or
   incoherent results, not just no speedup.

8. **Q: Why does Case 3 include both 5-thread and 40-thread runs?**
   A: The smaller run is a baseline and the larger run probes the boundary.
   The 40-thread case is executed, but this provider/model combination did
   not fail visibly; a production design should measure grounding and
   coverage across representative workloads before choosing map-reduce.

9. **Q: What does `tier3_advanced_multiagent_patterns` cover that this
   series deliberately doesn't?**
   A: Domain-expertise-driven multi-agent splits, where different
   sub-domains genuinely need different tools/knowledge -- a fourth
   failure-mode category this closing notebook names but doesn't
   demonstrate, left to that tier's deeper coverage.

10. **Q: Why is "it still produced an answer" not sufficient evidence
    that a single agent handled Case 3 correctly?**
    A: A plausible-sounding but ungrounded answer is a worse failure than
    an obvious one, since nothing signals the problem -- the same
    determinism-of-trust lesson from Part 1 of
    `agent_context_engineering.ipynb`, applied here to context volume
    instead of output schema.

## Assignments

1. Re-run Case 1 with 8 diffs instead of 4 and confirm the speedup ratio
   holds roughly proportional -- does it plateau at some point (hint:
   `max_workers` and real API rate limits both matter)?
2. The notebook already ran Case 2's comparison 3 times above. Looking at
   those three runs (not a new one), write a one-paragraph honest
   conclusion about how reliable the softening effect is empirically --
   then propose a concrete change to the prompt wording (e.g. "critique
   this" instead of "critique this as harshly as you can") that you'd
   expect to make contamination show up more reliably, and explain why.
3. (No new real API calls required) Design, in a markdown cell, a
   map-reduce agent architecture for Case 3 at 40-thread scale: how would
   you group threads for the "map" step, what would each summary need to
   preserve, and what would the "reduce" step's prompt need to explicitly
   ask for to avoid losing the same grounding Case 3 showed degrading?
4. Go back to notebook 01's incident-investigation agent and notebook
   02's release agent -- for each, argue (markdown only, referencing this
   notebook's three failure modes) whether either of them is actually a
   disguised Case 1, 2, or 3 candidate at larger real-world scale (e.g.
   investigating an incident across 50 microservices, or releasing 10
   independent services at once), and if so, what would need to change.